In [1]:
# Project-relative paths (supports notebooks launched from the project root or notebooks/).
from pathlib import Path
import sys

_PROJECT_CANDIDATES = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (p for p in _PROJECT_CANDIDATES if (p / "data" / "raw" / "mongo_test.test-result.json").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Launch Jupyter from the project root or notebooks/ directory.")
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "artifacts" / "models"
PREPROCESSING_DIR = PROJECT_ROOT / "artifacts" / "preprocessing"
SPLITS_DIR = PROJECT_ROOT / "artifacts" / "splits"
for _path in (DATA_PROCESSED, MODELS_DIR, PREPROCESSING_DIR, SPLITS_DIR):
    _path.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))


In [2]:
import json
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

# ──────────────────────────────────────────────────────────
# 1. LOAD DATA
# ──────────────────────────────────────────────────────────

def load_data(path: str) -> pd.DataFrame:
    """
    Load from JSON array file.
    Adjust this function if your data is CSV or JSON Lines.
    """
    with open(path, "r") as f:
        records = json.load(f)
    return pd.DataFrame(records)

# ──────────────────────────────────────────────────────────
# 2. CLEAN & TYPE-FIX
# ──────────────────────────────────────────────────────────

def clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["batchSize"] = df["batchSize"].astype(int)
    return df

# ──────────────────────────────────────────────────────────
# 3. SPLIT FEATURES / TARGETS
# ──────────────────────────────────────────────────────────

NUMERICAL_FEATURES   = ["batchSize", "lingerMs", "billSize"]
CATEGORICAL_FEATURES = ["compressionType"]
FEATURES             = NUMERICAL_FEATURES + CATEGORICAL_FEATURES
TARGETS              = ["throughput"]


def split_xy(df: pd.DataFrame):
    X = df[FEATURES]
    y = df[TARGETS]
    return X, y

# ──────────────────────────────────────────────────────────
# 4. BUILD SKLEARN PREPROCESSOR
# ──────────────────────────────────────────────────────────

def build_preprocessor() -> ColumnTransformer:
    """
    - Numerical columns  → StandardScaler  (zero mean, unit variance)
    - compressionType    → OneHotEncoder   (none, snappy, lz4, gzip)

    Note for GPR: also normalize targets separately (see normalize_targets).
    """
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), NUMERICAL_FEATURES),
            ("cat", OneHotEncoder(
                categories=[["none", "snappy", "lz4", "gzip"]],
                sparse_output=False,
                handle_unknown="ignore"
            ), CATEGORICAL_FEATURES),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


def get_feature_names(preprocessor: ColumnTransformer) -> list[str]:
    num_names = NUMERICAL_FEATURES
    cat_names = list(
        preprocessor
        .named_transformers_["cat"]
        .get_feature_names_out(CATEGORICAL_FEATURES)
    )
    return num_names + cat_names

# ──────────────────────────────────────────────────────────
# 5. TARGET NORMALIZATION  (needed for GPR)
# ──────────────────────────────────────────────────────────

class TargetScaler:
    """
    Thin wrapper so we can inverse-transform predictions easily.
    Fits one StandardScaler per target column.
    """
    def __init__(self):
        self.scalers = {t: StandardScaler() for t in TARGETS}

    def fit_transform(self, y: pd.DataFrame) -> np.ndarray:
        cols = []
        for t in TARGETS:
            scaled = self.scalers[t].fit_transform(y[[t]])
            cols.append(scaled)
        return np.hstack(cols)

    def transform(self, y: pd.DataFrame) -> np.ndarray:
        cols = []
        for t in TARGETS:
            scaled = self.scalers[t].transform(y[[t]])
            cols.append(scaled)
        return np.hstack(cols)

    def inverse_transform(self, y_scaled: np.ndarray) -> pd.DataFrame:
        cols = {}
        for i, t in enumerate(TARGETS):
            cols[t] = self.scalers[t].inverse_transform(
                y_scaled[:, i].reshape(-1, 1)
            ).flatten()
        return pd.DataFrame(cols)
# ──────────────────────────────────────────────────────────
# 6. MAIN PIPELINE
# ──────────────────────────────────────────────────────────

def preprocess(path: str, test_size: float = 0.2, random_state: int = 42):
    """
    Full preprocessing pipeline.

    Returns
    -------
    X_train, X_test : np.ndarray  — scaled feature matrices
    y_train, y_test : pd.DataFrame — raw targets  (for RF / XGBoost)
    y_train_s, y_test_s : np.ndarray — scaled targets (for GPR)
    feature_names : list[str]
    preprocessor : fitted ColumnTransformer
    target_scaler : fitted TargetScaler
    """

    # Load & clean
    df = load_data(path)
    df = clean(df)

    # Describe the dataset
    print("=" * 55)
    print(f"Dataset shape    : {df.shape}")
    print(f"\nFeatures used    : {FEATURES}")
    print(f"Targets          : {TARGETS}")
    print(f"\ncompression_type distribution:")
    print(df["compressionType"].value_counts().to_string())
    print(f"\nbatchSize unique : {sorted(df['batchSize'].unique())}")
    print(f"lingerMs range   : {df['lingerMs'].min()} – {df['lingerMs'].max()}")
    print(f"billSize unique  : {sorted(df['billSize'].unique())}")
    print(f"\nThroughput stats :\n{df['throughput'].describe().round(2).to_string()}")
    print("=" * 55)

    # Features / targets
    X, y = split_xy(df)

    # Train / test split  (stratify not used — continuous targets)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    # Fit preprocessor on train, apply to both splits
    preprocessor = build_preprocessor()
    X_train_s = preprocessor.fit_transform(X_train)
    X_test_s  = preprocessor.transform(X_test)

    feature_names = get_feature_names(preprocessor)
    print(f"\nFinal feature names ({len(feature_names)}):")
    print(feature_names)
    print(f"\nX_train : {X_train_s.shape}  |  X_test : {X_test_s.shape}")
    print(f"y_train : {y_train.shape}   |  y_test : {y_test.shape}")

    # Target scaling  (for GPR)
    target_scaler = TargetScaler()
    y_train_s = target_scaler.fit_transform(y_train)
    y_test_s  = target_scaler.transform(y_test)

    return (
        X_train_s, X_test_s,
        y_train, y_test,
        y_train_s, y_test_s,
        feature_names,
        preprocessor,
        target_scaler,
    )
results = preprocess(str(DATA_RAW / "mongo_test.test-result.json"))

import json

# Reload and clean the original file
df_all = load_data(str(DATA_RAW / "mongo_test.test-result.json"))
df_all = clean(df_all)

# Export to JSON
df_all.to_json(str(DATA_PROCESSED / "cleaned_data.json"), orient="records", indent=2)
print("Saved!")

(
    X_train_s, X_test_s,
    y_train, y_test,
    y_train_s, y_test_s,
    feature_names,
    preprocessor,
    target_scaler,
) = results
df_scaled = pd.DataFrame(X_train_s, columns=feature_names)
df_scaled.to_json(str(DATA_PROCESSED / "scaled_data.json"), orient="records", indent=2)
print("saved!!")

Dataset shape    : (1248, 5)

Features used    : ['batchSize', 'lingerMs', 'billSize', 'compressionType']
Targets          : ['throughput']

compression_type distribution:
compressionType
none      312
lz4       312
snappy    312
gzip      312

batchSize unique : [np.int64(16384), np.int64(32768), np.int64(49152), np.int64(65536)]
lingerMs range   : 5 – 505
billSize unique  : [np.int64(309000), np.int64(618000), np.int64(927000)]

Throughput stats :
count      1248.00
mean      93217.72
std       26954.28
min       53636.52
25%       70158.18
50%       89263.39
75%      106286.01
max      187841.95

Final feature names (7):
['batchSize', 'lingerMs', 'billSize', 'compressionType_none', 'compressionType_snappy', 'compressionType_lz4', 'compressionType_gzip']

X_train : (998, 7)  |  X_test : (250, 7)
y_train : (998, 1)   |  y_test : (250, 1)
Saved!
saved!!


In [3]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel, ConstantKernel

In [4]:
kernel = ConstantKernel(1.0) * Matern(nu=1.5) + WhiteKernel(noise_level=1e-2)

gpr_model = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=10,
    normalize_y=True,
    random_state=42,
)

# GPR needs scaled targets and a 1D array
gpr_model.fit(X_train_s, y_train_s.ravel())

print("✅ GPR trained!")
print(f"   Optimized kernel: {gpr_model.kernel_}")

✅ GPR trained!
   Optimized kernel: 0.912**2 * Matern(length_scale=1.18, nu=1.5) + WhiteKernel(noise_level=0.0832)


In [5]:
import numpy as np
import pandas as pd
from itertools import product
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Predict — GPR returns predictions + standard deviation (uncertainty)
y_pred_gpr_scaled, y_std = gpr_model.predict(X_test_s, return_std=True)

# Inverse transform back to real units
y_pred_gpr = target_scaler.scalers["throughput"].inverse_transform(
    y_pred_gpr_scaled.reshape(-1, 1)
).ravel()

true = y_test.values[:, 0]

print(f"\nTHROUGHPUT — GPR")
print(f"  R²   : {r2_score(true, y_pred_gpr):.4f}")
print(f"  MAE  : {mean_absolute_error(true, y_pred_gpr):.2f} docs/sec")
print(f"  RMSE : {np.sqrt(mean_squared_error(true, y_pred_gpr)):.2f} docs/sec")
print(f"\n  Avg prediction uncertainty : ±{y_std.mean():.4f} (scaled units)")


THROUGHPUT — GPR
  R²   : 0.8211
  MAE  : 7565.13 docs/sec
  RMSE : 11876.04 docs/sec

  Avg prediction uncertainty : ±0.3482 (scaled units)


In [6]:
BATCH_SIZE_VALUES  = [16384, 32768, 49152, 65536]
LINGER_MS_VALUES   = list(range(5, 506, 20))
COMPRESSION_VALUES = ["none", "snappy", "lz4", "gzip"]

def recommend_gpr(bill_size, top_n=3):
    # Build all possible config combinations
    combos = list(product(BATCH_SIZE_VALUES, LINGER_MS_VALUES, COMPRESSION_VALUES))
    grid   = pd.DataFrame(combos, columns=["batchSize", "lingerMs", "compressionType"])
    grid["billSize"] = bill_size

    # Preprocess and predict with uncertainty
    X_grid = preprocessor.transform(grid)
    preds_scaled, stds = gpr_model.predict(X_grid, return_std=True)

    # Inverse transform to real units
    preds = target_scaler.scalers["throughput"].inverse_transform(
        preds_scaled.reshape(-1, 1)
    ).ravel()

    grid["predicted_throughput"] = preds
    grid["uncertainty"]          = stds

    # Rank by throughput
    grid = grid.sort_values("predicted_throughput", ascending=False)

    # Print top N
    print(f"\n{'='*50}")
    print(f"GPR  |  billSize = {bill_size:,}")
    print(f"{'='*50}")
    for rank, (_, row) in enumerate(grid.head(top_n).iterrows(), 1):
        print(f"\nRank #{rank}")
        print(f"  batchSize       : {int(row['batchSize'])}")
        print(f"  lingerMs        : {int(row['lingerMs'])} ms")
        print(f"  compressionType : {row['compressionType']}")
        print(f"  → throughput    : {row['predicted_throughput']:,.0f} docs/sec")
        print(f"  → uncertainty   : ±{row['uncertainty']:.4f} (scaled units)")

In [7]:
recommend_gpr(bill_size=309000)
recommend_gpr(bill_size=609000)
recommend_gpr(bill_size=909000)


GPR  |  billSize = 309,000

Rank #1
  batchSize       : 65536
  lingerMs        : 365 ms
  compressionType : lz4
  → throughput    : 177,481 docs/sec
  → uncertainty   : ±0.3226 (scaled units)

Rank #2
  batchSize       : 65536
  lingerMs        : 385 ms
  compressionType : lz4
  → throughput    : 177,442 docs/sec
  → uncertainty   : ±0.3226 (scaled units)

Rank #3
  batchSize       : 65536
  lingerMs        : 345 ms
  compressionType : lz4
  → throughput    : 176,130 docs/sec
  → uncertainty   : ±0.3231 (scaled units)

GPR  |  billSize = 609,000

Rank #1
  batchSize       : 65536
  lingerMs        : 405 ms
  compressionType : none
  → throughput    : 161,644 docs/sec
  → uncertainty   : ±0.3247 (scaled units)

Rank #2
  batchSize       : 65536
  lingerMs        : 425 ms
  compressionType : none
  → throughput    : 161,405 docs/sec
  → uncertainty   : ±0.3248 (scaled units)

Rank #3
  batchSize       : 65536
  lingerMs        : 385 ms
  compressionType : none
  → throughput    : 161,2

In [8]:
import joblib

In [9]:
joblib.dump(gpr_model, str(MODELS_DIR / "gpr_model.pkl"))

['D:\\Amin\\My Python Projects\\MFT-Kafka-Ensemble-Optimization\\artifacts\\models\\gpr_model.pkl']